In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import json, zipfile
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, confusion_matrix
from baseline import calculate_resilience_cost
SEED = 42
np.random.seed(SEED)

# --- Load data ---
df = pd.read_csv("data/train.csv", index_col=0)
test_df = pd.read_csv("data/test.csv",  index_col=0)
cost_matrix = pd.read_csv("data/cost_matrix.csv", index_col=0).values

display(df)

In [ ]:
# Data inspection
display(df.describe())



In [ ]:
sns.barplot(data=df["alert"]) 

In [ ]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder ,OrdinalEncoder
categories=[["green", "yellow", "orange", "red"]]
# --- Encode target and split ---
# enc = OrdinalEncoder(categories=categories, dtype=int)
enc = LabelEncoder()
display(enc)
enc.fit(df.loc[:, ["alert"]])
display(df.loc[:, ["alert"]])
y = enc.transform(df.loc[:, ["alert"]]).ravel()
display(y)
X = df.drop("alert", axis=1)
X_final = test_df
display(X.corr())
display(df.info())

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

pipe = make_pipeline(StandardScaler())
numeric_features = ["magnitude", "depth", "cdi", "mmi", "sig"]
numeric_transformer = Pipeline(
    # steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    steps=[("scaler", StandardScaler())]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        # ("cat", categorical_transformer, categorical_features),
    ]
)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.compose import TransformedTargetRegressor

class_weight = {
        0: 0.075,
        1: 0.35,
        2: 0.5,
        3: 0.125,
    }

clf = Pipeline(
    # steps=[("preprocessor", preprocessor), ("classifier", SVC(C=700, class_weight="balanced"))])
    steps=[("preprocessor", preprocessor), ("classifier", SVC(C=700, class_weight=class_weight))]
)
clf

In [ ]:
def resilience_scorer(y_true, y_pred):
    cm = confusion_matrix(enc.inverse_transform(y_true), enc.inverse_transform(y_pred))
    rci = calculate_resilience_cost(cm, cost_matrix)
    return rci

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.metrics import make_scorer
# We use resilience_score rather than accuracy, since this is what will be competed upon
resilience_score = make_scorer(resilience_scorer, greater_is_better=False)



# The class weights are very interesting, they seem to make a big impact since recall can be slightly controlled 

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

param_grid = {
    'classifier__gamma': ["scale"],
    'classifier__kernel': [ "rbf"],
    'classifier__C': np.linspace(1, 2000)}
    # 'classifier_class_weigths_': [{"}
    # 'classifier__C': np.linspace(1, 1)}

grid_search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    scoring=resilience_score)


grid_search.fit(X_train, y_train)
pred = grid_search.predict(X_val)
print(classification_report(y_val, pred))

In [ ]:
display(grid_search.best_params_)
display(grid_search.best_score_)
clf = grid_search


In [ ]:
# # --- Scale features ---
# scaler = MinMaxScaler().fit(X_train)
# X_train, X_val, X_test = (
#     scaler.transform(X_train),
#     scaler.transform(X_val),
#     scaler.transform(test_df),
# )

In [ ]:

# # # --- Train baseline model ---
# clf = KNeighborsClassifier(n_neighbors=10)
# clf.fit(X_train, y_train)
# clf


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay


# --- Validate ---
y_val_pred = clf.predict(X_val)
f1 = f1_score(y_val, y_val_pred, average="macro")
cm = confusion_matrix(enc.inverse_transform(y_val), enc.inverse_transform(y_val_pred))
rci = calculate_resilience_cost(cm, cost_matrix)

cm_display = ConfusionMatrixDisplay(cm, display_labels=["green", "orange", "red", "yellow"]).plot()
print(f"F1 (macro): {f1:.3f}")
# print("Confusion matrix:\n", cm)
print(f"Resilience Cost: {rci:.2f}")

In [ ]:
from baseline import create_submission

best_params = grid_search.best_params_
final_clf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", SVC(
            C=best_params["classifier__C"],
            gamma=best_params["classifier__gamma"],
            kernel=best_params["classifier__kernel"],
            class_weight=class_weight
        ))
    ]
)

final_clf.fit(X, y)
y_final = final_clf.predict(X_final)
display(y_final)
create_submission(enc.inverse_transform(y_final))